In [9]:
from openai import OpenAI
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI()

vector_stores = client.vector_stores.list()

for vector_store in vector_stores.data:
  print(vector_store.name, vector_store.id)

vector_store = vector_stores.data[0]
print(vector_store.name)

rule_demo vs_6a7e70f5d54081918aa389bfa2166faf
rule_demo vs_6a7e6f1f86ec8191a47937bca4f70975
ChatGPT Clone Store vs_6916bddd46d0819195176b24bb2b0788
rule_demo


In [10]:
query = "방송통신대에는 어떤 부속시설이 있나요?"

# 벡터스토어 검색 수행
search_results = client.vector_stores.search(
  vector_store_id=vector_store.id,
  query=query,
  max_num_results=5
)

for result in search_results.data:
  print(result.score, result.content[0].text)



In [ ]:
files = client.files.list()
for f in files.data:
  print(f.id, f.filename)

file = files.data[0]
print(file)

file-Mr8Y39arcqSsL7zeERuyLH 한국방송통신대학교 학칙.pdf
file-KDqENryyDaMvVWHqKdJJg5 한국방송통신대학교 학칙.pdf
file-Hk2RF99djdfsqPzm9my8JQ 한국방송통신대학교 학칙.pdf
file-KtUGmwD8RYJEiyYDtQNdPW 한국방송통신대학교 학칙.pdf
file-EH9fGmsagohL1Lg58eZqdH 한국방송통신대학교 학칙.pdf
file-7SsNYDWPA1x7R28kE6o727 한국방송통신대학교 학칙.pdf
file-TZo3hXpBAxycQDfxGf32Cw 한국방송통신대학교 학칙.docx
file-2ACgrjmLxqPS82rmNanVDT 한국방송통신대학교 학칙.pdf
file-TSswV857iuNQ8okWA0yQyl9x step_metrics.csv
file-a1asGVA48iFmLQl51yRV4V6x FOR_FINE_20240315_1638.jsonl
file-bMYBNpyEpowpHAWipGIoQJ0Z FOR_FINE_20240315_1630.jsonl
file-667aIGqROkB1ic56ZMYRx6xn FOR_FINE_20240315_1508.jsonl
FileObject(id='file-Mr8Y39arcqSsL7zeERuyLH', bytes=941634, created_at=1766554950, filename='한국방송통신대학교 학칙.pdf', object='file', purpose='assistants', status='processed', expires_at=None, status_details=None)


In [8]:
# 파일 업로드
file = client.files.create(
    file=open("/content/한국방송통신대학교 학칙.doc", "rb"),
    purpose="assistants"
)

In [11]:
# Vector Store 재생성
client.vector_stores.delete(vector_store.id)
vector_store = client.vector_stores.create(
  name="knou_rules",
  metadata={
    "description": "한국방송통신대학교 학칙"
   },
  chunking_strategy={
    'type': 'static',
    'static': {
      'max_chunk_size_tokens': 800,
      'chunk_overlap_tokens': 200
    }
  }, # 또는 auto
  file_ids = [file.id] #웹 또는 client.files.list() 이용해 확인
)

print(vector_store)

VectorStore(id='vs_6a83fde4d0588191b0319bdf0d8304c1', created_at=1787035109, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=1, total=1), last_active_at=1787035109, metadata={'description': '한국방송통신대학교 학칙'}, name='knou_rules', object='vector_store', status='in_progress', usage_bytes=0, expires_after=None, expires_at=None, description=None)


In [12]:
import time

def wait_for_vectorization(vector_store_id: str, file_id: str, poll_interval: int = 2):
    while True:
        vs_file = client.vector_stores.files.retrieve(
            vector_store_id=vector_store_id,
            file_id=file.id,
        )
        print("현재 상태:", vs_file.status)

        if vs_file.status in ("completed", "failed", "cancelled"):
            return vs_file

        time.sleep(poll_interval)

final_vs_file = wait_for_vectorization(vector_store.id, file.id)
print("최종 상태:", final_vs_file.status)

현재 상태: completed
최종 상태: completed


In [14]:
# 파일 업로드
file = client.files.create(
    file=open("/content/한국방송통신대학교 학칙.doc", "rb"),
    purpose="assistants"
)

# Vector Store에 파일 추가
client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=file.id
)

VectorStoreFile(id='file-JqRp7aQTxEViHGgyKtCGmp', created_at=1787035919, last_error=None, object='vector_store.file', status='in_progress', usage_bytes=0, vector_store_id='vs_6a83fde4d0588191b0319bdf0d8304c1', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))

In [15]:
question = "방송통신대에는 어떤 본교 부속시설이 있나요?"

response = client.responses.create(
    model="gpt-4.1-mini",
    input=question,
    instructions="당신은 학사정보 상담사입니다. 제공된 학칙 문서를 기반으로 정확한 답변을 제공해주세요. 답변 마지막에는 참조한 문서에서 해당하는 부분의 내용을 <출처 : > 안에 표시해주세요. 예 : <출처 : 제 9 조>",
    temperature=0.2,
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store.id],
            "max_num_results": 5
        }
    ],
    include=["file_search_call.results"]
)

print(response.output_text)
print(response.output[0].results[0].text)

방송통신대학교 본교에는 다음과 같은 부속시설이 있습니다.

1. 평생교육원  
2. 종합교육연수원  
3. 교양교육원  
4. 역사기록관  
5. 국제협력단  
6. 산학협력단  
7. 인권센터 (신설)  
8. 교원양성지원센터 (신설)  

또한 본교에는 교육기본시설로 중앙도서관이 있으며, 연구시설로는 원격교육혁신연구원과 통합인문학연구소가 있습니다.

이 부속시설들의 장은 관련 단과대학, 학과(부)의 장이 겸직하되, 부속시설의 특수성을 반영하여 총장이 별도로 임명할 수 있습니다.

참고로, 지역대학과 디지털미디어센터도 부속시설로 두고 있습니다.

이 내용은 방송통신대학교 학칙 제22조(부속시설 등)에 명시되어 있습니다.<출처 : 제22조>
② 「대학도서관진흥법」 제 6조 제1항에 따라 본교에 교육기본시설로 중앙도서관을 둔다 .
③ 본교에 다음 각 호의 부속시설을 둔다 .
1. 평생교육원
2. 종합교육연수원
3. 교양교육원
4. 역사기록관
5. 삭제(2024.1.19.)
6. 국제협력단
7. 산학협력단
8. 인권센터(신설)
9. 삭제(2026.4.8.)
10. 교원양성지원센터(신설 2024.1.19.)
④ 본교에 다음 각 호의 연구시설을 둔다 .
1. 원격교육혁신연구원 (개정 2022.12.20., 2026.4.8.)
2. 통합인문학연구소
⑤ 각 부속시설 등의 장은 관련 단과대학 , 학과(부)의 장이 겸직하되 , 부속시설 등의 특수성을 반영하여 총장이 별도로 
임명할 수 있다 .
⑥ 총장은 「대학 설립 ·운영 규정」 제 4조제1항에 따른 연구시설에 대해서는 2년마다 해당시설의 운영 실적을 평가하여 한국방송통신대학교 학칙
법제처 5 국가법령정보센터
존속 또는 폐지 여부를 결정한다 . 이 경우 평가에 관한 세부 사항은 총장이 정한다 .
제23조(지역대학) ① 입학, 수업, 시험, 장학의 학업 지원과 학생 활동을 지원하고 지역사회의 평생교육 진흥에 기여하기 위하여 
지역대학을 둔다.
② 지역대학에 학장을 두며 , 학장은 소관 지역대학의 업무를 총괄하고 소속직원을 지

In [ ]:
from vespa.application import Vespa

app = Vespa(url="http://localhost:8080")
response = app.feed_data_point(
    schema="doc",
    data_id="id:ns:doc::1",
    fields={
        "id": 1,
        "title": "1조",
        "description": "(목적) 이 학칙은 한국방송통신대학교의 교육목표를...",
        "vector": [0.1, 0.2, ..., 0.8]  # 768차원
    }
)